# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q wordcloud Faker
    print('Packages installed.')

    # Download NLTK data
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print('NLTK data downloaded.')


# Word Cloud & N-Grams In-Depth

This notebook explores text visualization techniques beyond simple frequency counts. We will generate a larger synthetic dataset, preprocess it, and visualize not just single words (unigrams) but also pairs (bigrams) and triplets (trigrams) to understand context.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.util import ngrams
from nltk.corpus import stopwords
from collections import Counter
import re
from faker import Faker
import random

# Ensure stopwords are available locally
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

sns.set_style('whitegrid')

## 1. Generate Synthetic Text Corpus
We will simulate a collection of customer feedback comments about a fictional coffee shop.

In [ ]:
fake = Faker()
Faker.seed(42)

coffee_adjectives = ['great', 'excellent', 'poor', 'bitter', 'sweet', 'cold', 'hot', 'delicious', 'stale']
staff_adjectives = ['friendly', 'rude', 'helpful', 'slow', 'fast', 'attentive', 'lazy']
products = ['latte', 'cappuccino', 'espresso', 'pastry', 'muffin', 'cake', 'tea']

def generate_review():
    template = random.choice([
        f"The {random.choice(products)} was {random.choice(coffee_adjectives)}.",
        f"Staff was {random.choice(staff_adjectives)}.",
        f"I had a {random.choice(coffee_adjectives)} {random.choice(products)} here.",
        f"Service was {random.choice(staff_adjectives)} but the {random.choice(products)} was {random.choice(coffee_adjectives)}."
    ])
    return template

reviews = [generate_review() for _ in range(100)]

df = pd.DataFrame({'text': reviews})
print(f"Total reviews: {len(df)}")
print(df.head())

## 2. Text Preprocessing
We need to remove stopwords (common words like 'the', 'is', 'and') to focus on meaningful content.

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return words

# Create a single list of all words
all_words = []
for review in df['text']:
    all_words.extend(preprocess(review))

print(f"Total words after cleaning: {len(all_words)}")
print(all_words[:10])

## 3. Visualizing N-Grams
N-grams are contiguous sequences of $n$ items. Bigrams (2 words) helps us capture phrases like "wait times" or "free wifi".

In [ ]:
def plot_ngrams(words, n, title):
    n_grams = ngrams(words, n)
    counter = Counter(n_grams)
    common = counter.most_common(10)
    
    labels = [' '.join(x[0]) for x in common]
    counts = [x[1] for x in common]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=counts, y=labels, palette='viridis')
    plt.title(title)
    plt.xlabel('Frequency')
    plt.show()

plot_ngrams(all_words, 1, 'Top 10 Unigrams (Single Words)')
plot_ngrams(all_words, 2, 'Top 10 Bigrams (Two-Word Phrases)')
plot_ngrams(all_words, 3, 'Top 10 Trigrams (Three-Word Phrases)')

## 4. Word Cloud
A word cloud emphasizes the most frequent words. We'll customize the color map.

In [ ]:
text_corpus = ' '.join(all_words)

wordcloud = WordCloud(
    width=800, 
    height=400, 
    background_color='white', 
    colormap='magma', 
    max_words=100
).generate(text_corpus)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Customer Feedback Word Cloud', fontsize=16)
plt.show()